# bert-cuda on Colab
Runtime > Change runtime type > **T4 GPU** first.

On your Mac, run `scripts/package_for_colab.sh` to create `bert-cuda.zip` (code + `data/`), then upload it below.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,driver_version --format=csv
!nvcc --version | tail -2

In [ ]:
from google.colab import files
files.upload()  # choose bert-cuda.zip
!unzip -q -o bert-cuda.zip

In [ ]:
%cd /content/bert-cuda
cap = !nvidia-smi --query-gpu=compute_cap --format=csv,noheader
arch = 'sm_' + cap[0].strip().replace('.', '')
print('building for', arch)
!make -j4 ARCH={arch}

In [ ]:
# Kernel unit tests (FP32 GEMMs, bit-exact fixed point, attention)
!make test

In [ ]:
# BERT on GPU vs. CPU reference: accuracy, latency, CUDA graphs, batching
!make run DATA=data

In [ ]:
# Fixed-point precision sweep for the FFN layers
!make sweep DATA=data

In [ ]:
# GEMM benchmark: 1 to 256 sentences, all kernels vs. cuBLAS
!make bench

In [ ]:
# Optional: Nsight Systems timeline (Nsight Compute counters are usually blocked on Colab)
!nsys profile -o results/timeline --force-overwrite true --trace cuda ./bin/bert_infer --data data --gemm reg32 --iters 20 > /dev/null
!nsys stats --report cuda_gpu_kern_sum results/timeline.nsys-rep | head -40

In [ ]:
!zip -qr results.zip results
files.download('results.zip')